# Agent Offline Evaluation — Quality Evaluators

Runs quality, RAG, and RAI evaluators against `aria-rm-briefing-agent` responses captured in `test_data.jsonl`. Each evaluator is an LLM-as-judge call against `CHAT_MODEL` on the hub, scoring how well aria's narrative answers the RM's question and how well it stays grounded in the underlying KB data.

In [1]:
import hashlib
import json
import os
import subprocess
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    CoherenceEvaluator,
    FluencyEvaluator,
    RelevanceEvaluator,
    GroundednessEvaluator,
    SimilarityEvaluator,
    ViolenceEvaluator,
    HateUnfairnessEvaluator,
    AzureOpenAIModelConfiguration,
    evaluate,
)
from dotenv import load_dotenv

## Environment

In [2]:
repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ['CHAT_MODEL']

SUB_ID = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'
AOAI_ENDPOINT    = f'https://aif-core-{SUFFIX}.services.ai.azure.com/'

## Model configuration

Keyless Entra auth — the evaluator's grader model is the same `CHAT_MODEL` deployment on the admin project's parent account (`aif-core-{suffix}`).

In [3]:
credential = DefaultAzureCredential()

# Python 3.13 + azure-ai-evaluation 1.16.x workaround: omit `credential` from
# AzureOpenAIModelConfiguration (SDK validates via isinstance(value, Any), which
# Python 3.13 made into a hard TypeError). Pass credential as a kwarg to each
# evaluator below instead. See 08-06-00 README for details.
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AOAI_ENDPOINT,
    azure_deployment=CHAT_MODEL,
)


## Load test data

In [4]:
lab_dir        = repo_root / '08-agents' / '08-06-agent-offline-evaluation'
test_data_path = lab_dir / 'test_data.jsonl'

test_records = []
with open(test_data_path) as f:
    for line in f:
        line = line.strip()
        if line:
            test_records.append(json.loads(line))

if not test_records:
    raise RuntimeError('test_data.jsonl is empty. Run 08-05-01 first.')

print(f'Loaded {len(test_records)} records')

Loaded 5 records


## Spot-check: single-row quality evaluators

In [5]:
coherence_eval = CoherenceEvaluator(model_config=model_config, credential=credential)
fluency_eval   = FluencyEvaluator(model_config=model_config, credential=credential)
relevance_eval = RelevanceEvaluator(model_config=model_config, credential=credential)

first = test_records[0]

print('Coherence :', coherence_eval(query=first['query'], response=first['response']))
print('Fluency   :', fluency_eval(query=first['query'], response=first['response']))
print('Relevance :', relevance_eval(query=first['query'], response=first['response'], context=first['context']))

Coherence : {'coherence': 5.0, 'gpt_coherence': 5.0, 'coherence_reason': 'The response is coherent because it logically organizes the requested information into clear sections, directly addresses all parts of the query, and presents data in an orderly and easy-to-follow manner, making the briefing comprehensive and accessible.', 'coherence_result': 'pass', 'coherence_threshold': 3, 'coherence_prompt_tokens': 1813, 'coherence_completion_tokens': 231, 'coherence_total_tokens': 2044, 'coherence_finish_reason': 'stop', 'coherence_model': 'gpt-4.1-mini-2025-04-14', 'coherence_sample_input': '[{"role": "user", "content": "{\\"query\\": \\"I have a 9am with the Berger family for their quarterly review. Brief me on their portfolio, any drift, recent activity, anything CRM has flagged, and the 2-3 things I should be ready to talk about.\\", \\"response\\": \\"Here is the briefing for the Berger Family Trust (UHNW Multi-Generation segment, RM Anna M\\\\u00fcller) for your 9am quarterly review:\\

## Batch evaluate: groundedness and similarity

In [ ]:
groundedness_results = evaluate(
    data=str(test_data_path),
    evaluators={
        'groundedness': GroundednessEvaluator(model_config=model_config, credential=credential),
        'similarity':   SimilarityEvaluator(model_config=model_config, credential=credential),
    },
    evaluator_config={
        'groundedness': {'column_mapping': {'query': '${data.query}', 'response': '${data.response}', 'context': '${data.context}'}},
        'similarity':   {'column_mapping': {'query': '${data.query}', 'response': '${data.response}', 'ground_truth': '${data.ground_truth}'}},
    },
)

print('Metrics:', groundedness_results.get('metrics', {}))

2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Finished 1 / 5 lines.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Average execution time for completed lines: 2.15 seconds. Estimated time for incomplete lines: 8.6 seconds.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Finished 2 / 5 lines.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Average execution time for completed lines: 1.1 seconds. Estimated time for incomplete lines: 3.3 seconds.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Finished 3 / 5 lines.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Average execution time for completed lines: 0.75 seconds. Estimated time for incomplete lines: 1.5 seconds.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Finished 4 / 5 lines.
2026-05-11 12:46:07 +0200 139025391265472 execution.bulk     INFO     Average execution time for co

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "groundedness_20260511_104605_177794"
Run status: "Completed"
Start time: "2026-05-11 10:46:05.177794+00:00"
Duration: "0:00:06.381183"

======= Combined Run Summary (Per Evaluator) =======

{
    "groundedness": {
        "status": "Completed",
        "duration": "0:00:06.381183",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "similarity": {
        "status": "Completed",
        "duration": "0:00:02.367265",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}


Metrics: {'groundedness.groundedness': 5.0, 'groundedness.gpt_groundedness': 5.0, 'similarity.similarity': 3.8, 'similarity.gpt_similarity': 3.8, 'groundedness.binary_aggregate': 1.0, 'similarity.binary_aggregate': 0.8}


/home/jp/development/corticalstack/foundry-nextgen/08-agents/08-06-agent-offline-evaluation/evaluation_helpers.py:107: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_df = df.style.applymap(highlight_scores, subset=existing_score_cols).hide(axis="index")
Class ViolenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class HateUnfairnessEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


2026-05-11 12:46:28 +0200 139024844965568 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:46:28 +0200 139024844965568 execution.bulk     INFO     Average execution time for completed lines: 3.28 seconds. Estimated time for incomplete lines: 0.0 seconds.
2026-05-11 12:46:28 +0200 139024844965568 execution          ERROR    1/5 flow run failed, indexes: [1], exception of index 1: Error while evaluating single input: HttpResponseError: (RequestThrottled) Received too many requests in a short amount of time. Retry again after 1 seconds.
Code: RequestThrottled
Message: Received too many requests in a short amount of time. Retry again after 1 seconds.


Run violence_20260511_104611_719499 failed with status 4.
Error: (InternalError) 20% of the batch run failed. (RequestThrottled) Received too many requests in a short amount of time. Retry again after 1 seconds.
Code: RequestThrottled
Message: Received too many requests in a short amount of time. Retry again after 1 seconds.
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "violence_20260511_104611_719499"
Run status: "Failed"
Start time: "2026-05-11 10:46:11.719499+00:00"
Duration: "0:00:16.423280"

azure.ai.evaluation._legacy._batch_engine._exceptions.BatchEngineRunFailedError: (InternalError) 20% of the batch run failed. (RequestThrottled) Received too many requests in a short amount of time. Retry again after 1 seconds.
Code: RequestThrottled
Message: Received too many requests in a short amount of time. Retry again after 1 seconds.

2026-05-11 12:46:34 +0200 139024853358272 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:46:34 +0200 139024853358272 execution.bulk     INFO     Average execution time for completed lines: 4.55 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "hate_unfairness_20260511_104611_720511"
Run status: "Completed"
Start time: "2026-05-11 10:46:11.720511+00:00"
Duration: "0:00:22.735070"

======= Combined Run Summary (Per Evaluator) =======

{
    "violence": {
        "status": "Failed",
        "duration": "0:00:16.423280",
        "completed_lines": 4,
        "failed_lines": 1,
        "log_path": null,
        "per_line_errors": {
            "1": "(RequestThrottled) Received too many requests in a short amount of time. Retry again after 1 seconds.\nCode: RequestThrottled\nMessage: Received too many requests in a short amount of time. Retry again after 1 seconds."
        },
        "error_message": "(SystemError) 20% of the batch run failed. (RequestThrottled) Received too many requests in a short amount of time. Retry again after 1 seconds.\nCode: RequestThrottled\nMessage: Received too many requests in a short amount of time. Retry again after 1 seconds.",
        "error_code": "FAILED_

## Display results

In [7]:
import sys
sys.path.insert(0, str(lab_dir))
from evaluation_helpers import display_metrics_summary, display_row_results

display_metrics_summary(groundedness_results.get('metrics', {}))
display_row_results(groundedness_results.get('rows', []), columns=['groundedness', 'similarity'])

### Aggregate Metrics Summary

#### RAG & Similarity Metrics

Metric,Score
groundedness.groundedness,5.00
groundedness.gpt_groundedness,5.00
similarity.similarity,3.80
similarity.gpt_similarity,3.80
groundedness.binary_aggregate,1.00
similarity.binary_aggregate,0.80


### Row-Level Results

#,Query,Groundedness,Similarity
1,I have a 9am with the Berger family for ...,5.000000,4.000000
2,Show me the Lindemann family office port...,5.000000,4.000000
3,Anything been written recently about AI ...,5.000000,4.000000
4,Summarise what is been happening on the ...,5.000000,5.000000
5,Get me the FINMA sustainability disclosu...,5.000000,2.000000


## RAI evaluators

Require `DefaultAzureCredential` and `evaluate_query=True` (breaking change since SDK 1.10.0).

In [8]:
azure_ai_project = PROJECT_ENDPOINT

rai_results = evaluate(
    data=str(test_data_path),
    evaluators={
        'violence':        ViolenceEvaluator(
                               azure_ai_project=azure_ai_project,
                               credential=credential,
                               evaluate_query=True,
                           ),
        'hate_unfairness': HateUnfairnessEvaluator(
                               azure_ai_project=azure_ai_project,
                               credential=credential,
                               evaluate_query=True,
                           ),
    },
    evaluator_config={
        'violence':        {'column_mapping': {'query': '${data.query}', 'response': '${data.response}'}},
        'hate_unfairness': {'column_mapping': {'query': '${data.query}', 'response': '${data.response}'}},
    },
)

display_metrics_summary(rai_results.get('metrics', {}))

### Aggregate Metrics Summary

#### Custom Metrics

Metric,Score
violence.violence_score,0.00
hate_unfairness.hate_unfairness_score,0.00
violence.violence_defect_rate,0.00
hate_unfairness.hate_unfairness_defect_rate,0.00
violence.binary_aggregate,0.80
hate_unfairness.binary_aggregate,1.00
